# Tutorial 01: QM/MM training data from any force field, and training

Alanine dipeptide in TIP3P water (Amber14). The QM region is the alanine side chain (CA, HA, CB, HB1–3),
cut from the backbone, with two hydrogen link atoms. This notebook runs the same steps as the scripts in this
folder; the [README](README.md) explains every option in more detail.

**Energy model.** FieldMACE predicts the QM energy **including** the interaction with the MM point charges;
everything else is the force field. The *embedding* (which charges, link atoms, cutoff and switching, origin) is
saved in `setup.json` and used again for ML/MM MD.

In [ ]:
import os, subprocess, sys
import numpy as np
from IPython.display import Image, display

QUICK = os.environ.get("FIELDMACE_TUTORIAL_QUICK") == "1"   # tiny settings for automatic checks
os.environ.setdefault("OMP_NUM_THREADS", "4")

def run(cmd, env=None):
    """Run a tutorial step and show its output (stops the notebook if the step fails)."""
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                       env={**os.environ, **(env or {})})
    lines = [l for l in p.stdout.splitlines() if "Warning" not in l and "warnings.warn" not in l]
    print("\n".join(lines[-60:]))
    if p.returncode:
        raise RuntimeError(f"step failed: {cmd}")

## Step 1: classical system and QM region

`01_prepare_system.py` reads any OpenMM input (PDB + ffxml as here, Amber prmtop, GROMACS/OPLS top, CHARMM psf),
selects the QM atoms, finds the QM–MM bonds that need link atoms, and equilibrates the system. Try
`--qm "index 6-15"` (more of the backbone) or `--cutoff 0` (all charges).

In [ ]:
run(f"python 01_prepare_system.py --equil-ps {0.2 if QUICK else 20}")

In [ ]:
import json
setup = json.load(open("work/setup.json"))
print(json.dumps(setup, indent=1))

## Step 2: sampling

Classical MD; every frame is written in FieldMACE's format with `frame_to_atoms` (QM + link atoms, MM point
charges in `atoms.info`, already switched by the cutoff function).

In [ ]:
run(f"python 02_sample_configurations.py --n-frames {6 if QUICK else 300} --stride-ps {0.02 if QUICK else 0.5}")

In [ ]:
import ase.io
frames = ase.io.read("work/frames.xyz", index=":")
a = frames[0]
print(a.get_chemical_symbols(), "(last two H are link atoms)")
print("info keys:", sorted(a.info))
n_mm = [len(f.info["mm_charges"]) for f in frames]
print("point charges per frame:", min(n_mm), "-", max(n_mm))
q = a.info["mm_charges"]; r = np.linalg.norm(a.info["mm_positions"], axis=1)
import matplotlib.pyplot as plt
plt.figure(figsize=(5, 3)); plt.scatter(r, q, s=4)
plt.xlabel("distance from the QM centroid (A)"); plt.ylabel("charge seen by FieldMACE (e)")
plt.title("cutoff 8 A, switched from 6 A"); plt.show()

## Step 3: DFT labels with PySCF

Energy (incl. the point-charge interaction), forces on QM + link atoms, forces on the point charges, in eV and Å.
This takes ~5 s per frame on a few cores. The full data set (300 frames, B3LYP/6-31G*) ships with the tutorial
in `data/*.xyz.gz`; here only a few frames are labelled again, into `work/`, to show how it works.

In [ ]:
run(f"python 03_label_with_pyscf.py --max-frames {2 if QUICK else 5} --basis {'sto-3g' if QUICK else '6-31g*'} --out work/demo_data --partial work/demo_partial.xyz")

In [ ]:
lab = ase.io.read("work/demo_partial.xyz", index=":")
x = lab[0]
print("REF_energy (eV):", x.info["REF_energy"])
print("REF_forces (eV/A), shape", np.shape(x.info["REF_forces"]))
print("REF_mm_forces (eV/A), shape", np.shape(x.info["REF_mm_forces"]))

## Step 4: training

Two models: `PerAtomFieldEMACE` (multipole expansion of the MM potential about every QM atom, recommended) and
`FieldEMACE` (one expansion about the QM centroid). On a GPU use `DEVICE=cuda` and more epochs/channels.

In [ ]:
env = {"EPOCHS": "2", "CHANNELS": "8"} if QUICK else {}
run("bash 04_train.sh", env=env)

## Step 5: evaluation

In [ ]:
run("python 05_evaluate.py")
display(Image("results/parity_ground_state.png"))

## Step 6 (optional): HDF5 for large data sets

`mace_preprocess_data` converts the data once (point charges and their forces included); training then reads
the HDF5 files.

In [ ]:
run("bash 06_hdf5.sh", env={"EPOCHS": "1"})

## Next

* Tutorial 03, part 1: ML/MM MD in OpenMM with `models/aladip_PerAtomFieldEMACE.model`.
* Tutorial 02: excited states.